In [25]:
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

# =========================
# 0.1 路径（你给的本地路径）
# =========================
PATH_FINAL = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_outputs.csv"
PATH_RAW   = "/Users/ronnygottheway/Desktop/美赛/data/raw/2026_MCM_Problem_C_Data.csv"

# 输出文件（建议都存到 cleaned_data 里）
OUT_MERGED_LONG   = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/merged_long_with_features.csv"
OUT_SCORED_WEEKLY = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/weekly_scores_faps.csv"


# =========================
# 0.2 通用函数：安全找列名（防止你列名略有不同）
# =========================
def find_col(df, candidates, required=True):
    """
    在 df.columns 里匹配候选列名（完全匹配优先；忽略大小写和空格）
    """
    cols = list(df.columns)
    norm = {c: re.sub(r"\s+", "", c).lower() for c in cols}

    for cand in candidates:
        cand_norm = re.sub(r"\s+", "", cand).lower()
        for c in cols:
            if norm[c] == cand_norm:
                return c

    # 再尝试：包含关系
    for cand in candidates:
        cand_norm = re.sub(r"\s+", "", cand).lower()
        for c in cols:
            if cand_norm in norm[c]:
                return c

    if required:
        raise KeyError(f"找不到列：{candidates}。当前列有：{cols[:30]} ...")
    return None


def percentile_rank(s: pd.Series):
    """
    在组内把数值转换为分位数(0~1)，用于“同周同季”标准化：
    J_norm = PercentileRank(mean_score | season,week)
    """
    return s.rank(pct=True, method="average")


def winsorize_series(s: pd.Series, p_low=0.01, p_high=0.99):
    """
    简单 winsorize：把极端值截断到分位点，防止粉丝票极端爆炸
    """
    lo = s.quantile(p_low)
    hi = s.quantile(p_high)
    return s.clip(lower=lo, upper=hi)


def make_age_group(age):
    """
    年龄分组（与你第三问一致的分段：<25, 25-34, 35-44, 45-54, 55+）
    """
    if pd.isna(age):
        return np.nan
    age = float(age)
    if age < 25:
        return "<25"
    elif age < 35:
        return "25-34"
    elif age < 45:
        return "35-44"
    elif age < 55:
        return "45-54"
    else:
        return "55+"


In [2]:
##1. 读入 raw（宽表）与 final（长表），定位 merge 键与关键变量列。

In [3]:
# ================
# 1.1 读入
# ================
raw = pd.read_csv(PATH_RAW)
final = pd.read_csv(PATH_FINAL)

print("raw shape:", raw.shape)
print("final shape:", final.shape)

# ================
# 1.2 找 merge 键（根据你常见字段命名进行候选匹配）
# ================
raw_name = find_col(raw, ["celebrity_name", "celebrity", "name", "contestant", "star_name"])
raw_season = find_col(raw, ["season", "Season"])
# raw 可能没有 week（宽表），week 在列名里

final_name = find_col(final, ["celebrity_name", "celebrity", "name", "contestant", "star_name"])
final_season = find_col(final, ["season", "Season"])
final_week = find_col(final, ["week", "Week"])

# 关键变量（final）
col_mean = find_col(final, ["mean_score", "mean score"])
col_pop  = find_col(final, ["popularity_oof", "popularity", "fan_votes", "votes"])
col_elim = find_col(final, ["if_elim", "if elim", "elim", "eliminated"])

print("Matched columns:")
print(" raw:", raw_name, raw_season)
print(" final:", final_name, final_season, final_week, col_mean, col_pop, col_elim)


raw shape: (421, 53)
final shape: (2777, 13)
Matched columns:
 raw: celebrity_name season
 final: celebrity_name season week mean_score popularity_oof if_elim


In [ ]:
##从 raw（宽表）提取“周评委结构特征”：judge_count、J_total、score_variance、争议度

In [4]:
# =========================
# 2.1 找出 raw 里所有 weekX_judgeY_score 列
# =========================
score_cols = [c for c in raw.columns if re.match(r"week\d+_judge\d+_score", str(c).lower())]
if len(score_cols) == 0:
    # 有些数据写法可能是 Week1_Judge1_Score 等，放宽匹配
    score_cols = [c for c in raw.columns if ("week" in str(c).lower() and "judge" in str(c).lower() and "score" in str(c).lower())]

print("Found score cols:", len(score_cols))

# =========================
# 2.2 把宽表转成长表：每行=选手-季-周-评委
# =========================
raw_long = raw[[raw_name, raw_season] + score_cols].melt(
    id_vars=[raw_name, raw_season],
    value_vars=score_cols,
    var_name="score_col",
    value_name="judge_score"
)

# 解析 week 与 judge_id
# 例如 week3_judge2_score -> week=3 judge=2
raw_long["week"] = raw_long["score_col"].str.extract(r"week(\d+)", expand=False).astype(float).astype("Int64")
raw_long["judge_id"] = raw_long["score_col"].str.extract(r"judge(\d+)", expand=False).astype(float).astype("Int64")

# 清理：只保留有分数的记录
raw_long["judge_score"] = pd.to_numeric(raw_long["judge_score"], errors="coerce")
raw_long = raw_long.dropna(subset=["week"])  # week 必须有
raw_long_valid = raw_long.dropna(subset=["judge_score"]).copy()

print("raw_long_valid shape:", raw_long_valid.shape)

# =========================
# 2.3 生成“周评委结构特征”（每行=选手-季-周）
# =========================
agg = raw_long_valid.groupby([raw_name, raw_season, "week"]).agg(
    judge_count_raw=("judge_score", "count"),
    J_total_raw=("judge_score", "sum"),
    score_mean_raw=("judge_score", "mean"),
    score_var_raw=("judge_score", "var"),
).reset_index()

# 争议标签：你可以按第三问设阈值（示例 0.68；你可改）
CONTROVERSY_THRESHOLD = 0.68
agg["controversy_flag"] = (agg["score_var_raw"].fillna(0) > CONTROVERSY_THRESHOLD).astype(int)

print(agg.head())


Found score cols: 44
raw_long_valid shape: (13783, 6)
  celebrity_name  season  week  judge_count_raw  J_total_raw  score_mean_raw  \
0      AJ McLean      29     1                3         18.0        6.000000   
1      AJ McLean      29     2                3         19.0        6.333333   
2      AJ McLean      29     3                3         21.0        7.000000   
3      AJ McLean      29     4                3         24.0        8.000000   
4      AJ McLean      29     5                3         24.0        8.000000   

   score_var_raw  controversy_flag  
0       0.000000                 0  
1       0.333333                 0  
2       0.000000                 0  
3       0.000000                 0  
4       0.000000                 0  


In [ ]:
##3. 从 raw 提取静态变量（年龄/行业/舞伴）并合并进 final（长表）

In [5]:
# =========================
# 3.1 在 raw 中找到静态列
# =========================
raw_age = find_col(raw, ["celebrity_age_during_season", "age", "celebrity_age"], required=False)
raw_ind = find_col(raw, ["celebrity_industry", "industry", "occupation"], required=False)
raw_partner = find_col(raw, ["ballroom_partner", "partner", "pro_dancer", "dancer_partner"], required=False)

static_cols = [raw_name, raw_season]
for c in [raw_age, raw_ind, raw_partner]:
    if c is not None:
        static_cols.append(c)

static = raw[static_cols].drop_duplicates().copy()

# 统一列名
rename_map = {raw_name: "celebrity_name", raw_season: "season"}
if raw_age: rename_map[raw_age] = "age"
if raw_ind: rename_map[raw_ind] = "industry"
if raw_partner: rename_map[raw_partner] = "ballroom_partner"
static = static.rename(columns=rename_map)

# age_group
if "age" in static.columns:
    static["age"] = pd.to_numeric(static["age"], errors="coerce")
    static["age_group"] = static["age"].apply(make_age_group)

print("static columns:", static.columns.tolist())
print(static.head())

# =========================
# 3.2 final 统一列名后 merge 静态信息
# =========================
final2 = final.copy()
final2 = final2.rename(columns={final_name: "celebrity_name", final_season: "season", final_week: "week"})
final2["week"] = pd.to_numeric(final2["week"], errors="coerce").astype("Int64")

final2 = final2.merge(static, on=["celebrity_name", "season"], how="left")

print("After merge static:", final2.shape)
print(final2[["celebrity_name", "season", "week"] + [c for c in ["age","age_group","industry","ballroom_partner"] if c in final2.columns]].head())


static columns: ['celebrity_name', 'season', 'age', 'industry', 'ballroom_partner', 'age_group']
      celebrity_name  season  age       industry     ballroom_partner  \
0      John O'Hurley       1   50  Actor/Actress  Charlotte Jorgensen   
1       Kelly Monaco       1   29  Actor/Actress            Alec Mazo   
2  Evander Holyfield       1   42        Athlete      Edyta Sliwinska   
3      Rachel Hunter       1   35          Model     Jonathan Roberts   
4      Joey McIntyre       1   32  Singer/Rapper      Ashly DelGrosso   

  age_group  
0     45-54  
1     25-34  
2     35-44  
3     35-44  
4     25-34  
After merge static: (2777, 17)
  celebrity_name  season  week   age age_group       industry  \
0  John O'Hurley       1     1  50.0     45-54  Actor/Actress   
1  John O'Hurley       1     2  50.0     45-54  Actor/Actress   
2  John O'Hurley       1     3  50.0     45-54  Actor/Actress   
3  John O'Hurley       1     4  50.0     45-54  Actor/Actress   
4  John O'Hurley       1

In [ ]:
##4. 把“周评委结构特征 agg”合并进 final（长表）

In [6]:
# agg 的键列统一
agg2 = agg.rename(columns={raw_name: "celebrity_name", raw_season: "season"}).copy()
agg2["week"] = agg2["week"].astype("Int64")

final3 = final2.merge(agg2, on=["celebrity_name", "season", "week"], how="left")

print("After merge weekly-judge features:", final3.shape)
print(final3[["celebrity_name","season","week","judge_count_raw","J_total_raw","score_var_raw","controversy_flag"]].head())


After merge weekly-judge features: (2777, 22)
  celebrity_name  season  week  judge_count_raw  J_total_raw  score_var_raw  \
0  John O'Hurley       1     1              3.0         20.0       0.333333   
1  John O'Hurley       1     2              3.0         26.0       0.333333   
2  John O'Hurley       1     3              3.0         24.0       1.000000   
3  John O'Hurley       1     4              3.0         21.0       1.000000   
4  John O'Hurley       1     5              3.0         27.0       0.000000   

   controversy_flag  
0               0.0  
1               0.0  
2               1.0  
3               1.0  
4               0.0  


In [ ]:
##5. 同周同季标准化：得到 J_norm、F_norm、I（进步）基础量

In [7]:
df = final3.copy()

# 确保关键列为数值
df["mean_score"] = pd.to_numeric(df[col_mean], errors="coerce")
df["popularity_oof"] = pd.to_numeric(df[col_pop], errors="coerce")
df["if_elim"] = pd.to_numeric(df[col_elim], errors="coerce").astype("Int64")

# =========================
# 5.1 J_norm：同 season-week 内对 mean_score 做分位数
# =========================
df["J_norm"] = df.groupby(["season","week"])["mean_score"].transform(percentile_rank)

# =========================
# 5.2 粉丝票：winsorize + log 压缩，再做分位数
# =========================
# 先 winsorize（同 season 内或全局都行；这里用全局）
df["V_wins"] = winsorize_series(df["popularity_oof"].fillna(0), 0.01, 0.99)
df["V_log"] = np.log1p(df["V_wins"])  # g(V)=log(1+V)

df["F_norm"] = df.groupby(["season","week"])["V_log"].transform(percentile_rank)

# =========================
# 5.3 进步项：I_raw, I_norm
# =========================
df = df.sort_values(["season","celebrity_name","week"]).reset_index(drop=True)

# 过去两周均值（rolling window=2, shift 1）
df["J_norm_lag1"] = df.groupby(["season","celebrity_name"])["J_norm"].shift(1)
df["J_norm_lag2"] = df.groupby(["season","celebrity_name"])["J_norm"].shift(2)
df["J_norm_prev2_mean"] = df[["J_norm_lag1","J_norm_lag2"]].mean(axis=1)

df["I_raw"] = df["J_norm"] - df["J_norm_prev2_mean"]
df["I_raw"] = df["I_raw"].fillna(0)  # 前两周无历史，进步记为0

df["I_norm"] = df.groupby(["season","week"])["I_raw"].transform(percentile_rank)

print(df[["season","week","celebrity_name","mean_score","J_norm","popularity_oof","F_norm","I_raw","I_norm"]].head(10))


   season  week     celebrity_name  mean_score    J_norm  popularity_oof  \
0       1     1  Evander Holyfield    2.204545  0.333333        0.155787   
1       1     2  Evander Holyfield    2.204545  0.333333        0.155787   
2       1     3  Evander Holyfield    2.204545  0.200000        0.155787   
3       1     1      Joey McIntyre    6.534091  0.666667        0.716246   
4       1     2      Joey McIntyre    6.534091  0.666667        0.716246   
5       1     3      Joey McIntyre    6.534091  0.600000        0.716246   
6       1     4      Joey McIntyre    6.534091  0.500000        0.716246   
7       1     5      Joey McIntyre    6.534091  0.333333        0.716246   
8       1     1      John O'Hurley    8.318182  1.000000        0.999956   
9       1     2      John O'Hurley    8.318182  1.000000        0.999956   

     F_norm     I_raw    I_norm  
0  0.333333  0.000000  0.583333  
1  0.333333  0.000000  0.583333  
2  0.400000 -0.133333  0.200000  
3  0.666667  0.000000  0.58

In [ ]:
##6. 公平校正（核心）：评委端残差化得到 J_star；粉丝端行业校正得到 F_star；争议项纳入

In [8]:
# =========================
# 6.1 准备建模数据（去掉缺失）
# =========================
model_df = df.copy()

# 必须存在的公平校正变量：age_group / industry / ballroom_partner
# 如果某列缺失，用 "Unknown" 填充以保证可训练
for c in ["age_group","industry","ballroom_partner"]:
    if c not in model_df.columns:
        model_df[c] = "Unknown"
    model_df[c] = model_df[c].astype("string").fillna("Unknown")

# =========================
# 6.2 评委端强校正：J_norm ~ age_group + industry + ballroom_partner
#     用 one-hot + 线性回归（可解释，写报告最舒服）
# =========================
X_cols_J = ["age_group","industry","ballroom_partner"]
y_J = model_df["J_norm"].fillna(0)

preprocess_J = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), X_cols_J)],
    remainder="drop"
)

reg_J = Pipeline(steps=[
    ("prep", preprocess_J),
    ("lr", LinearRegression())
])

reg_J.fit(model_df[X_cols_J], y_J)

model_df["J_hat"] = reg_J.predict(model_df[X_cols_J])
model_df["J_star"] = model_df["J_norm"] - model_df["J_hat"]

# =========================
# 6.3 粉丝端轻校正：F_norm ~ industry（只校正行业，不把粉丝“洗太干净”）
# =========================
X_cols_F = ["industry"]
y_F = model_df["F_norm"].fillna(0)

preprocess_F = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), X_cols_F)],
    remainder="drop"
)

reg_F = Pipeline(steps=[
    ("prep", preprocess_F),
    ("lr", LinearRegression())
])

reg_F.fit(model_df[X_cols_F], y_F)

model_df["F_hat_ind"] = reg_F.predict(model_df[X_cols_F])
model_df["F_star"] = model_df["F_norm"] - model_df["F_hat_ind"]

# =========================
# 6.4 争议项：score_var_raw -> C_norm（同周分位数）
# =========================
model_df["score_var_raw"] = pd.to_numeric(model_df["score_var_raw"], errors="coerce").fillna(0)
model_df["C_norm"] = model_df.groupby(["season","week"])["score_var_raw"].transform(percentile_rank)

# 争议项强度（你可调小：0.05~0.15）
# 正号表示：争议越大，粉丝端略加成（贴合第三问“粉丝关注争议”）
GAMMA_CONTROVERSY = 0.08
model_df["C_star"] = GAMMA_CONTROVERSY * (model_df["C_norm"] - 0.5)  # 中心化，避免整体抬升

print(model_df[["season","week","celebrity_name","J_norm","J_hat","J_star","F_norm","F_hat_ind","F_star","C_norm","C_star"]].head())


   season  week     celebrity_name    J_norm     J_hat    J_star    F_norm  \
0       1     1  Evander Holyfield  0.333333  0.496248 -0.162915  0.333333   
1       1     2  Evander Holyfield  0.333333  0.496248 -0.162915  0.333333   
2       1     3  Evander Holyfield  0.200000  0.496248 -0.296248  0.400000   
3       1     1      Joey McIntyre  0.666667  0.454050  0.212616  0.666667   
4       1     2      Joey McIntyre  0.666667  0.454050  0.212616  0.666667   

   F_hat_ind    F_star  C_norm  C_star  
0   0.602156 -0.268823    1.00   0.040  
1   0.602156 -0.268823    0.50   0.000  
2   0.602156 -0.202156    0.20  -0.024  
3   0.579577  0.087090    0.75   0.020  
4   0.579577  0.087090    1.00   0.040  


In [ ]:
##7.动态权重 + 综合分 S + 保护条款 + 每周淘汰预测

In [9]:
scored = model_df.copy()

# =========================
# 7.1 动态权重函数
# =========================
def get_weights(week):
    # 你可以按赛季实际周数微调
    if pd.isna(week):
        return (0.50, 0.35, 0.15)
    w = int(week)
    if w <= 3:
        return (0.60, 0.30, 0.10)
    elif w <= 7:
        return (0.50, 0.35, 0.15)
    else:
        return (0.45, 0.45, 0.10)

weights = scored["week"].apply(get_weights)
scored["wJ"] = weights.apply(lambda x: x[0])
scored["wF"] = weights.apply(lambda x: x[1])
scored["wI"] = weights.apply(lambda x: x[2])

# =========================
# 7.2 进步项用 I_norm（已是分位数），也可残差化但通常没必要
# =========================
scored["I_star"] = scored["I_norm"] - 0.5  # 中心化，让进步是“加分/减分对称”

# 粉丝贡献上限（防极端）：把 F_star 截断到 [-0.25, +0.25]（可调）
scored["F_star_clipped"] = scored["F_star"].clip(-0.25, 0.25)

# =========================
# 7.3 综合分（未加保护条款）
# =========================
scored["S_raw"] = (
    scored["wJ"] * scored["J_star"] +
    scored["wF"] * scored["F_star_clipped"] +
    scored["wI"] * scored["I_star"] +
    scored["C_star"]
)

# =========================
# 7.4 保护条款：技术底线惩罚
# 技术底线：当周 J_norm 处于后10% -> 施加惩罚项 penalty
# =========================
# 先计算当周 J_norm 的分位（其实 J_norm 自己就是分位，但这里用它）
scored["tech_bottom10_flag"] = (scored["J_norm"] <= 0.10).astype(int)

PENALTY_TECH_BOTTOM10 = 0.08  # 可调：0.05~0.15
scored["S_final"] = scored["S_raw"] - PENALTY_TECH_BOTTOM10 * scored["tech_bottom10_flag"]

# =========================
# 7.5 按周预测淘汰：每个 season-week 最低 S_final 的人 pred_elim=1
# 如果某周真实淘汰人数>1，可改成 bottom-k
# =========================
def mark_bottom_k(group, k=1):
    group = group.copy()
    group = group.sort_values("S_final", ascending=True)
    group["pred_elim"] = 0
    group.iloc[:k, group.columns.get_loc("pred_elim")] = 1
    return group

# 先估计每周真实淘汰人数（用 if_elim）
elim_count = scored.groupby(["season","week"])["if_elim"].sum().reset_index(name="true_elim_k")
# 合并回去
scored = scored.merge(elim_count, on=["season","week"], how="left")
scored["true_elim_k"] = scored["true_elim_k"].fillna(1).astype(int).clip(lower=1)

# 用真实淘汰人数做 bottom-k（更公平评估）
out_list = []
for (s, w), g in scored.groupby(["season","week"], dropna=False):
    k = int(g["true_elim_k"].iloc[0]) if len(g) else 1
    out_list.append(mark_bottom_k(g, k=k))

scored2 = pd.concat(out_list, ignore_index=True)

print(scored2[["season","week","celebrity_name","S_final","pred_elim","if_elim","true_elim_k","tech_bottom10_flag"]].head(20))


    season  week     celebrity_name   S_final  pred_elim  if_elim  \
0        1     1      Trista Sutter -0.187950          1        0   
1        1     1  Evander Holyfield -0.124416          0        0   
2        1     1      Rachel Hunter  0.009492          0        0   
3        1     1      John O'Hurley  0.103317          0        0   
4        1     1      Joey McIntyre  0.182030          0        0   
5        1     1       Kelly Monaco  0.296295          0        0   
6        1     2  Evander Holyfield -0.164416          1        0   
7        1     2      Trista Sutter -0.161283          0        1   
8        1     2      Rachel Hunter -0.010508          0        0   
9        1     2      John O'Hurley  0.109984          0        0   
10       1     2      Joey McIntyre  0.202030          0        0   
11       1     2       Kelly Monaco  0.302962          0        0   
12       1     3  Evander Holyfield -0.292396          1        1   
13       1     3      Rachel Hunte

In [ ]:
##8. 回测评估（是否“更合理”）+ 公平性指标（年龄/舞伴依赖是否下降）

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

eval_df = scored2.dropna(subset=["if_elim"]).copy()

y_true = eval_df["if_elim"].astype(int)
y_pred = eval_df["pred_elim"].astype(int)

print("=== Elimination prediction metrics ===")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

# 8.1 每周命中率：真实淘汰者中有多少被预测为淘汰
weekly_hit = eval_df.groupby(["season","week"]).apply(
    lambda g: ( (g["if_elim"]==1) & (g["pred_elim"]==1) ).sum() / max(1, (g["if_elim"]==1).sum())
).reset_index(name="weekly_recall")

print("\nWeekly recall summary:")
print(weekly_hit["weekly_recall"].describe())

# =========================
# 8.2 公平性：年龄相关性 & 分组均值
# =========================
if "age" in eval_df.columns:
    eval_df["age"] = pd.to_numeric(eval_df["age"], errors="coerce")
    corr_age = eval_df[["age","S_final"]].dropna().corr().iloc[0,1]
    corr_age_Jnorm = eval_df[["age","J_norm"]].dropna().corr().iloc[0,1]
    print("\n=== Fairness: age correlation ===")
    print("corr(age, J_norm) :", corr_age_Jnorm)
    print("corr(age, S_final):", corr_age)

if "age_group" in eval_df.columns:
    group_stats = eval_df.groupby("age_group").agg(
        n=("S_final","count"),
        S_final_mean=("S_final","mean"),
        J_norm_mean=("J_norm","mean"),
        elim_rate=("if_elim","mean")
    ).reset_index()
    print("\n=== Fairness: age_group stats ===")
    print(group_stats.sort_values("age_group"))

# =========================
# 8.3 舞伴依赖：舞伴组间差异对比（J_norm vs S_final）
# =========================
if "ballroom_partner" in eval_df.columns:
    partner_stats = eval_df.groupby("ballroom_partner").agg(
        n=("S_final","count"),
        S_final_mean=("S_final","mean"),
        J_norm_mean=("J_norm","mean")
    ).reset_index()

    # 用组均值的标准差衡量“依赖强弱”（标准差越大，说明舞伴差异越影响分数）
    std_S = partner_stats.loc[partner_stats["n"]>=5, "S_final_mean"].std()
    std_J = partner_stats.loc[partner_stats["n"]>=5, "J_norm_mean"].std()

    print("\n=== Dependence: partner mean dispersion (n>=5) ===")
    print("std of partner mean J_norm :", std_J)
    print("std of partner mean S_final:", std_S)


=== Elimination prediction metrics ===
Accuracy: 0.9186172128195895
Precision: 0.6021220159151194
Recall: 0.7491749174917491
F1: 0.6676470588235294
Confusion matrix:
 [[2324  150]
 [  76  227]]

Weekly recall summary:
count    335.000000
mean       0.574627
std        0.483668
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: weekly_recall, dtype: float64

=== Fairness: age correlation ===
corr(age, J_norm) : -0.42359829039941743
corr(age, S_final): -0.18715092455523447

=== Fairness: age_group stats ===
  age_group    n  S_final_mean  J_norm_mean  elim_rate
0     25-34  936      0.026704     0.620622   0.086538
1     35-44  713     -0.004051     0.522413   0.119215
2     45-54  367     -0.028677     0.440388   0.160763
3       55+  265     -0.058038     0.314247        0.2
4       <25  489      0.065620     0.725299    0.04908
5   Unknown    7     -0.025383     0.435436   0.142857

=== Dependence: partner mean dispersion (n>=5) =

/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_7410/2622441350.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_hit = eval_df.groupby(["season","week"]).apply(


In [ ]:
##9. 保存“可写进第四问报告”的结果表（长表 + 每周排名表）

In [11]:
# 9.1 保存完整长表
scored2.to_csv(OUT_MERGED_LONG, index=False)
print("Saved:", OUT_MERGED_LONG)

# 9.2 生成每周排名表
weekly_rank = scored2.copy()
weekly_rank["rank_S_final"] = weekly_rank.groupby(["season","week"])["S_final"].rank(ascending=False, method="min")
weekly_rank = weekly_rank.sort_values(["season","week","rank_S_final"]).reset_index(drop=True)

weekly_rank.to_csv(OUT_SCORED_WEEKLY, index=False)
print("Saved:", OUT_SCORED_WEEKLY)


Saved: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/merged_long_with_features.csv
Saved: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/weekly_scores_faps.csv


In [ ]:
##可视化

In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# Paths
# =========================
PATH_WEEKLY = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/weekly_scores_faps.csv"
FIG_DIR = "/Users/ronnygottheway/Desktop/美赛/outputs/figures_faps"

os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(PATH_WEEKLY)

# Basic type cleaning
for c in ["season", "week"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

for c in ["S_final", "J_norm", "age"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Ensure key columns exist
required_cols = ["season","week","celebrity_name","S_final","J_norm"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in weekly file: {missing}. "
                     f"Available: {list(df.columns)}")

print("Loaded:", df.shape)
print("Saved figures to:", FIG_DIR)


Loaded: (2777, 48)
Saved figures to: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps


In [ ]:
#------------------------------------------------



##可视化





##1) 新机制评分结果：按周的 S_final 分布（Boxplot）：展示每周在新机制下选手整体得分分布（波动、收敛、后期分化等）。

In [13]:
# -------------------------
# Plot 1: S_final distribution by week
# -------------------------
tmp = df.dropna(subset=["week", "S_final"]).copy()
weeks = sorted(tmp["week"].dropna().unique().tolist())

data_by_week = [tmp.loc[tmp["week"] == w, "S_final"].values for w in weeks]

plt.figure()
plt.boxplot(data_by_week, labels=[str(w) for w in weeks], showfliers=False)
plt.title("Distribution of Final Score (S_final) by Week")
plt.xlabel("Week")
plt.ylabel("S_final")
plt.tight_layout()

outpath = os.path.join(FIG_DIR, "fig1_sfinal_boxplot_by_week.png")
plt.savefig(outpath, dpi=200)
plt.close()

print("Saved:", outpath)


Saved: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig1_sfinal_boxplot_by_week.png


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_7410/242927095.py:10: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_by_week, labels=[str(w) for w in weeks], showfliers=False)


In [ ]:
##2) 新机制评分结果：单赛季 Top-N 选手的 S_final 轨迹（Line chart）。展示新机制下“强者稳定性/逆袭/波动”，非常适合写第四问“机制效果”。

In [19]:
# -------------------------
# Plot 2: S_final trajectories for top contestants in a chosen season
# -------------------------

df = pd.read_csv(PATH_WEEKLY)

for c in ["season", "week"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["S_final"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# =========================
# Plot: S_final trajectories with names
# =========================
season_to_plot = int(df["season"].dropna().max())   # 或者手动写 34
top_n = 8

sub = df[df["season"] == season_to_plot].dropna(subset=["week", "S_final"]).copy()
final_week = int(sub["week"].max())

# Top-N based on final week S_final
sub_final = sub[sub["week"] == final_week].copy()
sub_final["rank_S"] = sub_final["S_final"].rank(ascending=False, method="min")
top_names = sub_final.sort_values("rank_S").head(top_n)["celebrity_name"].tolist()

plt.figure(figsize=(10, 6))  # 更宽一点，方便放图例

for name in top_names:
    g = sub[sub["celebrity_name"] == name].sort_values("week")
    plt.plot(g["week"], g["S_final"], marker="o", linewidth=1.8, label=name)

plt.title(f"S_final Trajectories of Top {top_n} Contestants (Season {season_to_plot})")
plt.xlabel("Week")
plt.ylabel("S_final")

# 图例放到右侧图外（不遮挡曲线）
plt.legend(
    title="Contestant",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True
)

plt.tight_layout()

outpath = os.path.join(FIG_DIR, f"fig2_sfinal_trajectories_season_{season_to_plot}_labeled.png")
plt.savefig(outpath, dpi=200, bbox_inches="tight")  # bbox_inches 保证图例也被保存进去
plt.close()

print("Saved:", outpath)


Saved: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig2_sfinal_trajectories_season_34_labeled.png


In [ ]:
##3) 新机制 vs 旧机制：同周排名变化 ΔRank（Histogram）。直观说明“新机制改变了谁的排名”（比如：更奖励进步/更抑制舞伴/年龄偏差等）。

In [31]:
# -------------------------
# Plot 3: Rank change histogram (new vs old)
# -------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ================== Paths ==================
DATA_PATH = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/weekly_scores_faps.csv"
OUT_PATH  = "/Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig3_rank_shift_3d_hist.png"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# ================== Load ==================
df = pd.read_csv(DATA_PATH)

# ================== Pick score columns (old/new) ==================
# 旧机制分数：优先用 J_norm（你之前图里就是它），其次尝试其它常见命名
old_candidates = ["J_norm", "J_hat", "mean_score", "judge_score", "score_mean_raw"]
# 新机制分数：优先用 S_final，其次尝试其它常见命名
new_candidates = ["S_final", "S_new", "score_new", "final_score"]

def pick_col(cands, cols):
    for c in cands:
        if c in cols:
            return c
    return None

old_col = pick_col(old_candidates, df.columns)
new_col = pick_col(new_candidates, df.columns)

if old_col is None or new_col is None:
    raise ValueError(
        f"Cannot find old/new score columns.\n"
        f"Found old_col={old_col}, new_col={new_col}.\n"
        f"Please check your file columns: {list(df.columns)}"
    )

# 必要列
need_cols = ["season", "week", "celebrity_name", old_col, new_col]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_PATH}: {missing}")

# ================== Clean + compute ranks inside (season, week) ==================
tmp = df[need_cols].copy()
tmp["season"] = pd.to_numeric(tmp["season"], errors="coerce")
tmp["week"]   = pd.to_numeric(tmp["week"], errors="coerce")
tmp[old_col]  = pd.to_numeric(tmp[old_col], errors="coerce")
tmp[new_col]  = pd.to_numeric(tmp[new_col], errors="coerce")
tmp = tmp.dropna(subset=["season", "week", "celebrity_name", old_col, new_col])

# 名次规则：分数越高名次越好 => rank 从 1 开始（descending）
tmp["rank_old"] = tmp.groupby(["season", "week"])[old_col].rank(ascending=False, method="average")
tmp["rank_new"] = tmp.groupby(["season", "week"])[new_col].rank(ascending=False, method="average")

tmp["delta_rank"] = tmp["rank_new"] - tmp["rank_old"]  # >0 表示新机制下名次更差

# ================== 3D histogram counts by Week x ΔRank bin ==================
weeks = np.sort(tmp["week"].unique())

# 用整数 bin（把 delta_rank 四舍五入到最近整数更贴合你之前的柱状图语义）
tmp["delta_rank_int"] = np.rint(tmp["delta_rank"]).astype(int)

bin_min = int(tmp["delta_rank_int"].min())
bin_max = int(tmp["delta_rank_int"].max())
bin_vals = np.arange(bin_min, bin_max + 1, 1)  # 每个整数一个柱

# counts: rows=weeks, cols=bin_vals
counts = np.zeros((len(weeks), len(bin_vals)), dtype=int)
for i, w in enumerate(weeks):
    sub = tmp.loc[tmp["week"] == w, "delta_rank_int"].values
    # 对齐到 bin_vals 顺序
    vc = pd.Series(sub).value_counts().reindex(bin_vals, fill_value=0).values
    counts[i, :] = vc

# ================== Plot 3D bars ==================
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")

xpos, ypos = np.meshgrid(np.arange(len(bin_vals)), np.arange(len(weeks)))
xpos = xpos.ravel()
ypos = ypos.ravel()
zpos = np.zeros_like(xpos)

dx = 0.8 * np.ones_like(zpos)
dy = 0.8 * np.ones_like(zpos)
dz = counts.ravel()

ax.bar3d(xpos, ypos, zpos, dx, dy, dz, shade=True)

# Labels (English)
ax.set_title(
    "3D Rank-Shift Histogram by Week\n"
    f"(ΔRank = rank_new - rank_old; old={old_col}, new={new_col})",
    pad=18
)
ax.set_xlabel("ΔRank Bin (Positive = Worse Rank Under New Mechanism)")
ax.set_ylabel("Week")
ax.set_zlabel("Count")

# Ticks
ax.set_xticks(np.arange(len(bin_vals)))
ax.set_xticklabels([str(int(v)) for v in bin_vals])
ax.set_yticks(np.arange(len(weeks)))
ax.set_yticklabels([str(int(w)) for w in weeks])

# View angle
ax.view_init(elev=22, azim=-55)

fig.tight_layout()
fig.savefig(OUT_PATH, dpi=220, bbox_inches="tight")
plt.close(fig)

print("Used columns:", {"old_col": old_col, "new_col": new_col})
print("Saved figure to:", OUT_PATH)


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_7410/3985195273.py:112: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


Used columns: {'old_col': 'J_norm', 'new_col': 'S_final'}
Saved figure to: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig3_rank_shift_3d_hist.png


In [ ]:
##4) 新机制 vs 旧机制：年龄偏差对比（两个散点图 + 拟合线 + 相关系数）。对应你已经算出的结论（年龄偏差显著减弱）

In [16]:
# -------------------------
# Plot 4A: Age vs J_norm (old)
# Plot 4B: Age vs S_final (new)
# -------------------------
tmp = df.dropna(subset=["age","J_norm","S_final"]).copy()

def corr(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if len(a) < 3:
        return np.nan
    return np.corrcoef(a, b)[0,1]

# 4A
x = tmp["age"].values
y = tmp["J_norm"].values
r = corr(x, y)

plt.figure()
plt.scatter(x, y, s=10)
# Fit line
m, b = np.polyfit(x, y, 1)
xx = np.linspace(np.min(x), np.max(x), 100)
plt.plot(xx, m*xx + b, linewidth=1)

plt.title(f"Age Bias in Old Score (J_norm) | corr = {r:.3f}")
plt.xlabel("Age")
plt.ylabel("J_norm")
plt.tight_layout()

outpath = os.path.join(FIG_DIR, "fig4a_age_vs_jnorm.png")
plt.savefig(outpath, dpi=200)
plt.close()
print("Saved:", outpath)

# 4B
y2 = tmp["S_final"].values
r2 = corr(x, y2)

plt.figure()
plt.scatter(x, y2, s=10)
m2, b2 = np.polyfit(x, y2, 1)
plt.plot(xx, m2*xx + b2, linewidth=1)

plt.title(f"Age Bias in New Score (S_final) | corr = {r2:.3f}")
plt.xlabel("Age")
plt.ylabel("S_final")
plt.tight_layout()

outpath = os.path.join(FIG_DIR, "fig4b_age_vs_sfinal.png")
plt.savefig(outpath, dpi=200)
plt.close()
print("Saved:", outpath)


Saved: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig4a_age_vs_jnorm.png
Saved: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig4b_age_vs_sfinal.png


In [ ]:
##5) 新机制 vs 旧机制：舞伴依赖变化（舞伴均值散点 + 组间离散度）。解释得到的结论（舞伴依赖显著下降），用图展示“舞伴均值从分散变集中”。

In [27]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ========= 1) Paths =========
DATA_PATH = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/weekly_scores_faps.csv"
OUT_PATH  = "/Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig5_partner_mean_old_vs_new_fixed.png"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# ========= 2) Load =========
df = pd.read_csv(DATA_PATH)

# 必要列检查（缺哪个就报错，避免“没输出但你不知道原因”）
need_cols = ["ballroom_partner", "J_norm", "S_final"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_PATH}: {missing}")

df["J_norm"]  = pd.to_numeric(df["J_norm"], errors="coerce")
df["S_final"] = pd.to_numeric(df["S_final"], errors="coerce")

# ========= 3) Partner-level aggregation (n>=5) =========
partner_stats = (
    df.dropna(subset=["ballroom_partner", "J_norm", "S_final"])
      .groupby("ballroom_partner")
      .agg(
          n_obs=("S_final", "count"),
          partner_mean_old=("J_norm", "mean"),
          partner_mean_new=("S_final", "mean"),
      )
      .reset_index()
)

n_threshold = 5
partner_stats = partner_stats[partner_stats["n_obs"] >= n_threshold].copy()

# 旧/新机制的“舞伴均值离散度”
std_old = partner_stats["partner_mean_old"].std(ddof=1)
std_new = partner_stats["partner_mean_new"].std(ddof=1)

# ========= 4) Plot =========
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(partner_stats["partner_mean_old"], partner_stats["partner_mean_new"], s=60)

title_line1 = f"Partner Effect: Mean Score by Partner (n>={n_threshold})"
title_line2 = f"std_old={std_old:.3f}, std_new={std_new:.3f}"
ax.set_title(title_line1 + "\n" + title_line2, fontsize=16, pad=12)

ax.set_xlabel("Partner Mean (Old: J_norm)", fontsize=13)
ax.set_ylabel("Partner Mean (New: S_final)", fontsize=13)

# 给长标题留空间，并防止保存时被裁切
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(OUT_PATH, dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved figure to:", OUT_PATH)


Saved figure to: /Users/ronnygottheway/Desktop/美赛/outputs/figures_faps/fig5_partner_mean_old_vs_new_fixed.png
